# Dense Evolution — Interactive Panel (ipywidgets, no Streamlit)

Runs entirely inside this notebook's cell output — no external tunnel/link, no leaving Colab. Sliders/dropdowns rebuild the circuit and re-render the panels in place.

Built on `dashboard_core` (the same refactored, tested backend the Streamlit dashboard uses), not the old `legacy/dash.py` — same interactivity, on the current bug-fixed foundation (includes Zero-Noise Extrapolation, added after the old ipywidgets panel was written).

**How to use:** run the two cells below in order. The second cell renders the panel directly under itself — change any control and click **▶ Esegui** to re-run.

Project: [github.com/tatopenn-cell/Dense-Evolution](https://github.com/tatopenn-cell/Dense-Evolution)

In [ ]:
# 1. Install from PyPI (dashboard_core + ipywidgets, no git clone needed)
!pip install -q -U "dense-evolution[dashboard,jax]" ipywidgets

In [ ]:
# 2. Build and display the interactive panel
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

import dashboard_core as dc

# ── Controls ───────────────────────────────────────────────────────────────
w_circuit = widgets.Dropdown(
    options=list(dc.QASM_LIBRARY.keys()), value='Bell |\u03a6+\u27e9', description='Circuito:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='420px'),
)
w_noise_model = widgets.Dropdown(
    options=['ideal', 'depolarizing', 'bitflip', 'phaseflip', 'amplitude_damping', 'combined'],
    value='ideal', description='Rumore:',
)
w_noise_p = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01, description='p:')
w_shots = widgets.IntSlider(value=512, min=50, max=5000, step=50, description='Shots:')
w_seed = widgets.IntText(value=42, description='Seed:')
w_engine = widgets.Dropdown(options=['dense', 'mps'], value='dense', description='Motore:')
w_zne = widgets.Checkbox(value=False, description='Abilita ZNE')
w_zne_healing = widgets.Checkbox(value=False, description='Healing predittivo')
w_run = widgets.Button(description='\u25b6 Esegui', button_style='primary')
w_status = widgets.HTML(value='')
w_out = widgets.Output()


def _on_run_clicked(_btn):
    w_status.value = '<span style="color:#00c8ff">\u23f3 Esecuzione...</span>'
    with w_out:
        clear_output(wait=True)
        res = dc.run_simulation(
            source_mode='Libreria Built-in', circuit_name=w_circuit.value, qasm_text='',
            noise_model=w_noise_model.value, noise_p=w_noise_p.value, shots=w_shots.value,
            seed=w_seed.value, use_float32=True, engine=w_engine.value,
        )
        fig = dc.build_panel_overview(res, df_vqe=None, corr_matrix=None,
                                       noise_model=w_noise_model.value, noise_p=w_noise_p.value)
        display(fig)
        plt.close(fig)  # avoid double-render (the inline backend also shows
                         # any figures left open at the end of a cell)

        if w_zne.value:
            if w_noise_model.value == 'ideal':
                print("\u26a0\ufe0f ZNE richiede un modello di rumore attivo, non 'ideal'.")
            else:
                mit_res = dc.run_mitigation_sweep(
                    'Libreria Built-in', w_circuit.value, '', w_noise_model.value,
                    w_noise_p.value, w_shots.value, w_seed.value,
                    healing_enabled=w_zne_healing.value,
                )
                fig_zne = dc.build_panel_mitigation(mit_res, res)
                display(fig_zne)
                plt.close(fig_zne)

    w_status.value = '<span style="color:#00ff9d">\u25cf Fatto</span>'


w_run.on_click(_on_run_clicked)

controls = widgets.VBox([
    w_circuit, w_noise_model, w_noise_p, w_shots, w_seed, w_engine,
    widgets.HBox([w_zne, w_zne_healing]),
    widgets.HBox([w_run, w_status]),
])
display(widgets.VBox([controls, w_out]))

---
Solo il pannello Overview + (se abilitato) il pannello ZNE per ora — VQE/MD/Mosaico/Fisica Stato/Hamiltoniana possono essere aggiunti allo stesso pattern (stessa `dashboard_core`, un'altra funzione `build_panel_*` e un altro controllo) se servono.